# Build & push paper-comparable OOD candidate pools

Run this **once** before `03_ood_em_baseline.ipynb`.

Creates on Drive (and optionally private Hub):

```text
data/ood/candidates/broad_text.jsonl          # ≥400 text prompts by default
data/ood/candidates/llava_mscoco_vqa.jsonl    # ≥400 VQA items (unique COCO images)
data/ood/images/coco/val2014/*.jpg
data/ood/candidate_pools.construction.json
```

**Sources (pinned at write time):**
- Text: `databricks/databricks-dolly-15k` HF dataset revision SHA frozen into each row
- Multimodal: Hugging Face `lmms-lab/VQAv2` validation (pinned revision) with embedded MSCOCO-linked images

This is a **paper-comparable reconstruction**, not the unreleased Gulati & Raval exact selection.

After this notebook:
1. Spot-check a few rows / images
2. In notebook 03, fill `INPUT_REVIEWER` / `INPUT_REVIEW_RECORD` and run the seal cell
3. Only then run generation


## 1. Runtime note

CPU is enough. GPU not required. Downloads ~300–500 COCO JPEGs (often 10–40 minutes).


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

DRIVE_PROJECT = Path('/content/drive/MyDrive/em-displacement-vlm')
HUB_NAMESPACE = 'rlogger'
PUSH_TO_HUB = True  # private dataset with candidates + images
BUILD_MANIFEST = True  # also write paper_comparable_ood_v1.jsonl (still unreviewed)
FORCE = True  # True: rebuild multimodal after partial failure; reuses text

SELECTION_SEED = 20260730  # must match notebook 03 OOD_SELECTION_SEED
N_TEXT_POOL = 400
N_MM_POOL = 400
MAX_WORKERS = 16

from google.colab import drive, userdata
drive.mount('/content/drive', force_remount=False)

for subdir in (
    'data/ood/candidates',
    'data/ood/images',
    'data/ood/cache',
    'runs',
):
    (DRIVE_PROJECT / subdir).mkdir(parents=True, exist_ok=True)

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
if not HF_TOKEN:
    raise SystemExit('Colab secret HF_TOKEN is required (read for Dolly; write if PUSH_TO_HUB).')
os.environ['HF_TOKEN'] = HF_TOKEN

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print('Drive project:', DRIVE_PROJECT)
print('PUSH_TO_HUB:', PUSH_TO_HUB, 'BUILD_MANIFEST:', BUILD_MANIFEST)


## 2. Clean checkout + install lightweight deps


In [ ]:
REPO_URL = 'https://github.com/rlogger/em-displacement-vlm.git'
REPO_DIR = Path('/content/em-displacement-vlm')
if REPO_DIR.exists():
    assert (REPO_DIR / '.git').is_dir(), f'{REPO_DIR} is not a git clone; Runtime → Restart session.'
    dirty = subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip()
    if dirty:
        raise SystemExit('Clone is dirty; Runtime → Restart session, then rerun this notebook.')
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', '--prune', 'origin', 'main'])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', '--detach', 'origin/main'])
else:
    subprocess.check_call(['git', 'clone', '--branch', 'main', '--single-branch', REPO_URL, str(REPO_DIR)])
%cd {REPO_DIR}
REPO_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Repository commit:', REPO_COMMIT)

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'datasets>=2.19', 'huggingface-hub>=0.23', 'pyyaml>=6.0',
])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR), '--no-deps'])
print('Package installed.')


Long step = HF VQAv2 image materialization. After a partial run (text OK, multimodal failed),
set `FORCE=True` to rebuild multimodal while reusing `broad_text.jsonl`. With full artifacts
present and `FORCE=False`, the script refuses to overwrite.


In [ ]:
import importlib
importlib.invalidate_caches()

cmd = [
    sys.executable, 'scripts/build_ood_candidate_pools.py',
    '--drive-project', str(DRIVE_PROJECT),
    '--selection-seed', str(SELECTION_SEED),
    '--n-text-pool', str(N_TEXT_POOL),
    '--n-mm-pool', str(N_MM_POOL),
    '--max-workers', str(MAX_WORKERS),
]
if FORCE:
    cmd.append('--force')
if BUILD_MANIFEST:
    cmd.append('--build-manifest')
if PUSH_TO_HUB:
    cmd.extend([
        '--push-to-hub',
        '--hub-repo', f'{HUB_NAMESPACE}/ood-candidates-paper-comparable-v1',
    ])

print('Running:', ' '.join(cmd))
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
with process:
    for line in process.stdout:
        print(line, end='')
if process.returncode:
    raise SystemExit(f'build_ood_candidate_pools.py failed with exit code {process.returncode}')
print('Builder finished.')


## 4. Sanity print for notebook 03


In [ ]:
TEXT = DRIVE_PROJECT / 'data' / 'ood' / 'candidates' / 'broad_text.jsonl'
MM = DRIVE_PROJECT / 'data' / 'ood' / 'candidates' / 'llava_mscoco_vqa.jsonl'
IMAGES = DRIVE_PROJECT / 'data' / 'ood' / 'images'
MANIFEST = DRIVE_PROJECT / 'data' / 'ood' / 'paper_comparable_ood_v1.jsonl'
CONSTRUCTION = DRIVE_PROJECT / 'data' / 'ood' / 'candidate_pools.construction.json'

def count_jsonl(path: Path) -> int:
    return sum(1 for line in path.read_text().splitlines() if line.strip())

assert TEXT.is_file(), TEXT
assert MM.is_file(), MM
assert CONSTRUCTION.is_file(), CONSTRUCTION
n_text = count_jsonl(TEXT)
n_mm = count_jsonl(MM)
n_images = sum(1 for p in IMAGES.rglob('*') if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
print('broad_text.jsonl rows:', n_text)
print('llava_mscoco_vqa.jsonl rows:', n_mm)
print('images on disk:', n_images)
print('construction record:', CONSTRUCTION)
if MANIFEST.exists():
    print('unreviewed OOD manifest rows:', count_jsonl(MANIFEST))
    print('NOTE: still must run validate_ood_manifest.py / notebook 03 seal with reviewer fields.')
else:
    print('No OOD manifest yet (BUILD_MANIFEST was False).')

assert n_text >= 150, n_text
assert n_mm >= 250, n_mm
assert n_images >= 250, n_images
print()
print('Next:')
print('  1. Spot-check JSONL lines and a few images on Drive.')
print('  2. Open notebooks/03_ood_em_baseline.ipynb with SEED=42.')
print('  3. In §5 set INPUT_REVIEWER / INPUT_REVIEW_RECORD, then seal.')
print('  4. Only after seal, run generation.')
